# Lekce 09 - Vzor metakognice


## Nastavení

Tento zápisník demonstruje návrhový vzor Metakognice za použití Microsoft Agent Framework.

**Požadavky:**
- Nasazení Azure OpenAI nakonfigurováno pomocí proměnných prostředí
- Ověření Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Co je metakognice?

Metakognice je **přemýšlení o přemýšlení**. V kontextu AI agentů to znamená vytvářet agenty, kteří mohou:

- **Sebereflektovat** své vlastní výstupy a proces uvažování
- **Detekovat chyby** a zotavit se s grácií místo tichého selhání
- **Hodnotit**, zda jsou jejich odpovědi kompletní a užitečné
- **Přizpůsobit** svou strategii, když počáteční přístup nefunguje (např. přechod na záložní systém)

Metakognitivní agent nejen odpovídá na otázky — sleduje svůj vlastní výkon a upravuje se za běhu.


## Primární a záložní nástroje

Běžným vzorem metakognice je **záložní strategie**. Agent nejprve vyzkouší primární nástroj; pokud selže (např. chyba 404), agent rozpozná selhání a transparentně přepne na záložní nástroj.

To odráží reálné systémy, kde primární služby nemusí být dostupné a agenti musí sami diagnostikovat problém, než zvolí alternativní cestu.

Níže definujeme dva nástroje pro vyhledávání letů:
- **Primární** — pokrývá Paříž, Tokio a Barcelonu
- **Záložní** — pokrývá Berlín, Sydney a New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent s vlastním sebehodnocením a obnovou po chybách

Níže uvedenému agentovi je zadáno, aby nejprve vyzkoušel primární letový systém, rozpoznal selhání a transparentně přešel na záložní systém. Po každé odpovědi krátce zhodnotí, zda plně odpověděl na uživatelovu otázku.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Vzor sebehodnocení

Dalším aspektem metakognice je **sebehodnocení**: samostatný agent (nebo tentýž agent při druhém průchodu) kontroluje odpověď z hlediska úplnosti, přesnosti a užitečnosti.

Níže vytvoříme agenta `ResponseEvaluator`, který hodnotí odpovědi cestovního agenta v třech dimenzích.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Shrnutí

V této lekci jste se naučili, jak vytvářet **metakognitivní agenty** pomocí Microsoft Agent Framework:

- **Sebe-reflexe**: Agenti, kteří sledují vlastní uvažování a transparentně komunikují, co se stalo.
- **Obnova po chybě s alternativami**: Vzor primárního + záložního nástroje, kde agent detekuje selhání (např. chyby 404) a automaticky zkouší alternativní zdroj.
- **Sebe-hodnocení**: Samostatný hodnotící agent, který hodnotí odpovědi z hlediska úplnosti, přesnosti a užitečnosti.

Tyto vzory dělají agenty robustnější, průhlednější a důvěryhodnější — což jsou kritické vlastnosti pro nasazení do produkce.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
